# Specialiserede Modeller — 1: Beslutningstræer & random forests

I Intro-ML byggede I neurale netværk — ML-verdenens schweizerkniv. Men en schweizerkniv
er sjældent det BEDSTE værktøj til noget som helst. I dette emne møder I modeller, der
er specialiserede — og vi starter med den model, der regerer på **tabeldata**:
**beslutningstræet** og dets storebror, **random forest**.

Dagens data: 8.124 svampe. Spørgsmålet: **tør du spise den?**

> Denne notebook er selvkørende og kræver kun viden fra Intro-ML — du kan tage emnets notebooks i den rækkefølge, du vil. Der er med vilje flere opgaver, end du kan nå: opgaver mærket **(ekstra)** er til de hurtige, og opgaver mærket **(find fejlen)** indeholder en bevidst fejl, som skal findes og rettes.

## Setup

In [ ]:
# Henter svampe-data fra GitHub (Plan B: upload mushrooms.csv via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/mushrooms.csv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

df = pd.read_csv("mushrooms.csv")
df.head()

# 1: Beslutningstræet — modellen der stiller spørgsmål

Kender I spillet *20 spørgsmål* (eller Akinator)? Man gætter hvad som helst ved at stille
ja/nej-spørgsmål, hvor hvert svar skærer halvdelen af mulighederne væk. Et
**beslutningstræ** er præcis dét — bare lært automatisk fra data:

> *Lugter svampen af noget?* → ja → *Er lugten mandel-agtig?* → nej → **GIFTIG**

Træet er på mange måder det modsatte af et neuralt netværk: det kan ikke det samme —
men det kan noget, intet netværk kan: **man kan læse det**.

## Dataene: 8.124 svampe

Hver række er en svamp, og `class` fortæller om den er **e**dible (spiselig) eller
**p**oisonous (giftig). Kig på tabellen ovenfor: ALLE kolonner er bogstavkoder —
kategorier, ikke tal! Fx `odor` (lugt): `a`=mandel, `l`=anis, `n`=ingen lugt,
`f`=modbydelig, `p`=skarp... og `cap-color` (hattens farve), `habitat` (voksested) osv.

In [ ]:
print(df.shape)
print(df["class"].value_counts())     # e = spiselig, p = giftig
print()
print(df["odor"].value_counts())      # lugt: n=ingen, f=modbydelig, a=mandel, l=anis, ...

## Tekst → tal: `pd.get_dummies`

Modeller spiser kun tal (Intro-ML-lektion nr. 1!). Men `odor` er ikke et tal, og det
ville være løgn at kode mandel=1, anis=2, ingen=3 — anis er jo ikke "dobbelt så meget"
som mandel! Løsningen er **one-hot encoding**: hver kategori får sin egen 0/1-kolonne
(`odor_a`, `odor_l`, `odor_n`,...). Det klarer `pd.get_dummies` i én linje:

In [ ]:
X = pd.get_dummies(df.drop(columns=["class"]))     # alle features, one-hot-kodet
y = (df["class"] == "p").astype(int)               # 1 = giftig, 0 = spiselig

print("før: ", df.shape, "— 22 tekstkolonner")
print("efter:", X.shape, "— kun 0/1-kolonner")
X.head(3)

## sklearn — ét mønster, hundrede modeller

Træer bygger vi ikke i PyTorch men i **scikit-learn** (sklearn) — bibliotekét for
"klassisk" ML. Og her er dagens vigtigste generelle færdighed: i sklearn ligner ALLE
modeller hinanden:

```python
model = EnEllerAndenModel()      # 1. opret
model.fit(X_traen, y_traen)      # 2. træn (én linje — intet træningsloop!)
model.predict(X_test)            # 3. forudsig
model.score(X_test, y_test)      # 4. mål accuracy
```

Lærer I det mønster, kan I bruge hundredvis af modeller. Ingen `zero_grad`, ingen
epoker — sklearn klarer alt indeni. Vi splitter i train/test som altid, og starter
med et bevidst LILLE træ (`max_depth=2` — højst 2 spørgsmål dybt):

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

X_traen, X_test, y_traen, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

trae = DecisionTreeClassifier(max_depth=2, random_state=42)
trae.fit(X_traen, y_traen)

print(f"accuracy på træningsdata: {trae.score(X_traen, y_traen):.1%}")
print(f"accuracy på testdata:     {trae.score(X_test, y_test):.1%}")

94 % med højst to spørgsmål! Og nu det magiske — lad os **læse** modellen:

In [ ]:
plt.figure(figsize=(13, 6))
plot_tree(trae,
          feature_names=X.columns,
          class_names=["spiselig", "giftig"],
          filled=True, fontsize=10)
plt.show()

**Sådan læses en kasse:** øverst står spørgsmålet (fx `odor_n <= 0.5` betyder "er
`odor_n` 0? — altså: lugter svampen af NOGET?"). `samples` er antal svampe i kassen,
`value` er [spiselige, giftige], og farven viser flertallet (jo mørkere, jo mere
"ren" er kassen — dvs. jo mere enig er svampene i kassen om svaret).

Træet har selv fundet ud af, at **lugt** er det bedste spørgsmål — præcis som en rigtig
svampeguide! At vælge det spørgsmål, der gør kasserne mest rene, er hele
træ-algoritmen. (Renhed måles med *gini* eller *entropi* — samme idé som når man i
20 spørgsmål vælger det spørgsmål, der skærer flest muligheder væk.)

## Hvor dybt skal træet være?

`max_depth` er træets vigtigste indstilling: for lavt = for dumt, for dybt = risiko for
udenadslære (overfitting — I kender fælden). Og hvilke spørgsmål gjorde mest nytte?
Det fortæller `feature_importances_`:

In [ ]:
trae4 = DecisionTreeClassifier(max_depth=4, random_state=42)
trae4.fit(X_traen, y_traen)

vigtighed = pd.Series(trae4.feature_importances_, index=X.columns)
vigtighed.sort_values().tail(8).plot(kind="barh")
plt.title("Hvilke spørgsmål gør mest nytte?")
plt.xlabel("vigtighed")
plt.show()

### Opgaver

##### Opgave 1.1
Kør `plot_tree`-cellen igen og følg denne svamp gennem træet **i hånden**: den lugter
af ingenting (`odor_n = 1`). Hvilken vej sendes den ved første spørgsmål, og hvad ender
træet med at gætte? Er du enig med træet — tør du spise den?

*Skriv jeres svar her:* $\dots$

##### Opgave 1.2
Skru på `max_depth`: prøv **1, 3, 5 og None** (None = ubegrænset dybde). Notér train-
og test-accuracy for hver. Hvornår rammer træet 100 % — og bliver test-accuracy nogen
sinde DÅRLIGERE af mere dybde her?

In [ ]:
trae = DecisionTreeClassifier(max_depth=1, random_state=42)   # ← prøv 1, 3, 5, None
trae.fit(X_traen, y_traen)
print(f"train: {trae.score(X_traen, y_traen):.1%}   test: {trae.score(X_test, y_test):.1%}")

##### Opgave 1.3
Udfyld de to `score`-kald, så både train- og test-accuracy måles for et træ med dybde 3.

In [ ]:
trae = DecisionTreeClassifier(max_depth=3, random_state=42)
trae.fit(X_traen, y_traen)
print("train:", trae.score(..., ...))
print("test: ", trae.score(..., ...))

##### Opgave 1.4 (find fejlen)
En kammerat praler: *"Min model rammer 100 % — og jeg behøvede ikke engang træne på
træningsdataene!"* Kig godt på koden. Hvad er der galt — og hvorfor er resultatet
værdiløst, selvom tallet er ægte nok?

In [ ]:
genial_model = DecisionTreeClassifier(random_state=42)
genial_model.fit(X_test, y_test)
print(f"accuracy: {genial_model.score(X_test, y_test):.1%} — GENIALT?!")

##### Opgave 1.5
Kør feature-importance-cellen igen. Hvilken kolonne vinder suverænt — og hvad betyder
den kolonne oversat til dansk svampejæger-sprog?

*Skriv jeres svar her:* $\dots$

##### Opgave 1.6
Træet uden dybdegrænse rammer **100 % på testdata**. I Intro-ML lærte I at være
mistænksomme ved perfekte tal. Diskutér: (a) hvorfor er 100 % normalt et alarmsignal,
(b) hvad gør, at det faktisk er ægte her, og (c) ville I spise en svamp, fordi modellen
siger "spiselig"?

*Skriv jeres svar her:* $\dots$

##### Opgave 1.7
Hvor meget kan træet uden sin superkraft? Fjern **hele lugte-kolonnen** fra dataene
(før `get_dummies`) og træn igen. Hvor god bliver test-accuracy nu — og hvor dyb skal
modellen være for at komme derop?

In [ ]:
df_uden_lugt = df.drop(columns=["odor"])
X2 = pd.get_dummies(df_uden_lugt.drop(columns=["class"]))
X2_traen, X2_test, y2_traen, y2_test = train_test_split(X2, y, test_size=0.2, random_state=42)

trae = DecisionTreeClassifier(max_depth=2, random_state=42)   # ← prøv flere dybder
trae.fit(X2_traen, y2_traen)
print(f"uden lugt: test-accuracy {trae.score(X2_test, y2_test):.1%}")

##### Opgave 1.8
Udfyld `predict`-kaldet og sammenlign modellens gæt for de 5 første test-svampe med
facit. (0 = spiselig, 1 = giftig.)

In [ ]:
trae = DecisionTreeClassifier(max_depth=4, random_state=42)
trae.fit(X_traen, y_traen)

gaet = trae.predict(...)
print("modellens gæt:", gaet[:5])
print("facit:        ", y_test.values[:5])

##### Opgave 1.9
Træer kan måle "renhed" på to måder: `criterion="gini"` (standard) eller
`criterion="entropy"` (20-spørgsmåls-målet). Prøv begge ved et par dybder — gør det
nogen forskel her? (Spoiler: sjældent. Men nu ved I, at knappen findes.)

In [ ]:
trae = DecisionTreeClassifier(max_depth=3, criterion="gini", random_state=42)   # ← prøv "entropy"
trae.fit(X_traen, y_traen)
print(f"test: {trae.score(X_test, y_test):.1%}")

##### Opgave 1.10 (ekstra)
Lav dybde-vs-accuracy-kurven: en for-løkke over dybderne 1–10, der gemmer train- og
test-accuracy i to lister og plotter dem sammen. Hvor flader kurverne ud — og åbner der
sig noget gab mellem dem?

In [ ]:
dybder = range(1, 11)
train_acc = []
test_acc = []
for dybde in dybder:
    trae = DecisionTreeClassifier(max_depth=dybde, random_state=42)
    trae.fit(X_traen, y_traen)
    train_acc.append(...)
    test_acc.append(...)

plt.plot(dybder, train_acc, "o-", label="train")
plt.plot(dybder, test_acc, "o-", label="test")
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.legend()
plt.show()

# 2: Random forest — visdom fra en flok

Ét træ har en akilleshæl: det tager sine beslutninger MEGET bogstaveligt. Er der fejl
eller støj i træningsdataene, lærer et dybt træ støjen udenad — klassisk overfitting.

Løsningen er lige så enkel som genial: **træn 100 forskellige træer og lad dem
stemme**. Hvert træ ser et tilfældigt udsnit af data og et tilfældigt udvalg af
features (derfor "random"), så de laver FORSKELLIGE fejl — og flertalsafstemningen
visker fejlene ud. Som at spørge 100 svampejægere i stedet for én excentrisk ekspert.

I sklearn er det bogstaveligt talt bare en anden klasse — samme fit/predict/score:

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

start = time.time()
skov = RandomForestClassifier(n_estimators=100, random_state=42)   # 100 træer
skov.fit(X_traen, y_traen)
print(f"test-accuracy: {skov.score(X_test, y_test):.1%}   (trænet på {time.time() - start:.2f} s)")

## Hvor skoven for alvor skinner: støjede data

Svampedataene er FOR pæne til at vise forskellen. Så lad os sabotere dem: vi "flipper"
10 % af træningslabels (som om nogen har tastet forkert i felterne) og ser, hvem der
holder hovedet koldt — ét dybt træ eller skoven:

In [ ]:
rng = np.random.default_rng(42)

y_stoej = y_traen.values.copy()
flip = rng.choice(len(y_stoej), size=int(0.10 * len(y_stoej)), replace=False)
y_stoej[flip] = 1 - y_stoej[flip]            # 10 % af svarene er nu FORKERTE

trae = DecisionTreeClassifier(random_state=42)
trae.fit(X_traen, y_stoej)
skov = RandomForestClassifier(n_estimators=100, random_state=42)
skov.fit(X_traen, y_stoej)

print("(målt på RENE testdata)")
print(f"ét dybt træ: {trae.score(X_test, y_test):.1%}")
print(f"skov:        {skov.score(X_test, y_test):.1%}")

Se DEN forskel! Træet lærte tastefejlene udenad og faldt til ~86 % — skoven stemte
sig igennem støjen og holder ~100 %. Det er derfor, random forests i den virkelige
verden (hvor data ALTID er støjede) er et af de mest brugte værktøjer overhovedet.

## Duellen: skov mod neuralt netværk

Sidste spørgsmål: hvad med jeres gamle ven, det neurale netværk? Lad os lade dem
kæmpe om svampene på fair vilkår — samme data, samme test:

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

X_traen_t = torch.tensor(X_traen.values.astype("float32"))
X_test_t = torch.tensor(X_test.values.astype("float32"))
y_traen_t = torch.tensor(y_traen.values.astype("float32"))
y_test_t = torch.tensor(y_test.values.astype("float32"))

netvaerk = nn.Sequential(nn.Linear(X.shape[1], 32), nn.ReLU(),
                         nn.Linear(32, 1), nn.Sigmoid())
tabsfunktion = nn.BCELoss()
optimizer = torch.optim.Adam(netvaerk.parameters(), lr=0.01)

start = time.time()
for epoke in range(200):                       # rytmen, som I kan i søvne
    optimizer.zero_grad()
    tab = tabsfunktion(netvaerk(X_traen_t).squeeze(), y_traen_t)
    tab.backward()
    optimizer.step()
nn_tid = time.time() - start

with torch.no_grad():
    nn_gaet = (netvaerk(X_test_t).squeeze() > 0.5).float()
print(f"neuralt netværk: {(nn_gaet == y_test_t).float().mean().item():.1%} på {nn_tid:.1f} s")

start = time.time()
skov = RandomForestClassifier(n_estimators=100, random_state=42)
skov.fit(X_traen, y_traen)
print(f"random forest:   {skov.score(X_test, y_test):.1%} på {time.time() - start:.2f} s")

Begge rammer (næsten) perfekt — men skoven er mange gange hurtigere, krævede NUL
hyperparameter-fifleri (ingen læringsrate, ingen epoker, ingen standardisering!), og
dens enkelt-træer kan læses. På tabeldata er træ-modeller derfor næsten altid det
første, man prøver. Netværkenes hjemmebane er billeder, lyd og sprog — det ser I i de
andre notebooks. **Det rigtige værktøj til det rigtige job.**

### Opgaver

##### Opgave 2.1
Hvor mange træer skal der til? Prøv `n_estimators` på **1, 5, 25 og 100** — på de
STØJEDE labels (`y_stoej`), hvor forskellen kan ses. Hvornår flader forbedringen ud?

In [ ]:
skov = RandomForestClassifier(n_estimators=1, random_state=42)   # ← prøv 1, 5, 25, 100
skov.fit(X_traen, y_stoej)
print(f"test: {skov.score(X_test, y_test):.1%}")

##### Opgave 2.2
Skru på sabotagen: prøv støj-andele på **10 %, 20 % og 30 %** og sammenlign træ vs skov
hver gang. Hvem knækker først — og hvor meget støj skal der til, før skoven for alvor
lider?

In [ ]:
andel = 0.10   # ← prøv 0.10, 0.20, 0.30
y_stoej2 = y_traen.values.copy()
flip = rng.choice(len(y_stoej2), size=int(andel * len(y_stoej2)), replace=False)
y_stoej2[flip] = 1 - y_stoej2[flip]

trae = DecisionTreeClassifier(random_state=42)
trae.fit(X_traen, y_stoej2)
skov = RandomForestClassifier(n_estimators=100, random_state=42)
skov.fit(X_traen, y_stoej2)
print(f"støj {andel:.0%}: træ {trae.score(X_test, y_test):.1%} | skov {skov.score(X_test, y_test):.1%}")

##### Opgave 2.3
Udfyld feature-importance-plottet for SKOVEN (den har også `feature_importances_` —
gennemsnittet over alle 100 træer). Sammenlign med det enkelte træs plot fra afsnit 1:
er vigtigheden mere spredt ud nu? Hvorfor giver det mening?

In [ ]:
skov = RandomForestClassifier(n_estimators=100, random_state=42)
skov.fit(X_traen, y_traen)

vigtighed = pd.Series(..., index=...)
vigtighed.sort_values().tail(10).plot(kind="barh")
plt.xlabel("vigtighed")
plt.show()

##### Opgave 2.4 (find fejlen)
En kammerat springer over, hvor gærdet er lavest, og fodrer skoven med den RÅ tabel —
uden `get_dummies`. Kør cellen, læs fejlbeskeden, og ret koden. Hvad prøver sklearn at
fortælle os?

In [ ]:
skov = RandomForestClassifier(n_estimators=100, random_state=42)
skov.fit(df.drop(columns=["class"]), y)
print(skov.score(df.drop(columns=["class"]), y))

##### Opgave 2.5
Vi kunne læse dybde-2-træet direkte i plot_tree. Kan man læse en skov på 100 træer,
der hver er 15 spørgsmål dybe? Hvad har vi OFRET for robustheden — og hvornår kan det
offer være et problem (tænk fx på en bank, der afviser dit lån)?

*Skriv jeres svar her:* $\dots$

##### Opgave 2.6
Kør NN-duellen ovenfor (hvis I ikke allerede har). Udfyld så sammenligningstabellen med
jeres egne ord — og kår en vinder *for netop dette datasæt*:

| | Random forest | Neuralt netværk |
|---|---|---|
| Accuracy | $\dots$ | $\dots$ |
| Træningstid | $\dots$ | $\dots$ |
| Krævede fifleri (lr, epoker, standardisering...)? | $\dots$ | $\dots$ |
| Kan modellen læses? | $\dots$ | $\dots$ |

##### Opgave 2.7 (ekstra)
Hvor LIDT træningsdata kan skoven nøjes med? Træn på tilfældige udsnit på **50, 200,
1000 og 6499** (alle) rækker og plot test-accuracy mod træningsstørrelse. Hvor lidt
skal der egentlig til her?

In [ ]:
stoerrelser = [50, 200, 1000, len(X_traen)]
resultater = []
for n in stoerrelser:
    udsnit = X_traen.sample(n=n, random_state=42)
    skov = RandomForestClassifier(n_estimators=100, random_state=42)
    skov.fit(udsnit, y_traen.loc[udsnit.index])
    resultater.append(skov.score(X_test, y_test))
    print(f"{n:5d} rækker: {resultater[-1]:.1%}")

plt.plot(stoerrelser, resultater, "o-")
plt.xscale("log")
plt.xlabel("antal træningsrækker")
plt.ylabel("test-accuracy")
plt.show()

##### Opgave 2.8 (ekstra)
I starter i praktik hos et firma, der giver jer et nyt TABELDATASÆT (kunder, salg,
sensorer — whatever) og spørger "kan I forudsige X?". Hvilken model prøver I FØRST, og
hvad er jeres begrundelse? Og hvornår ville I i stedet gribe til et neuralt netværk?

*Skriv jeres svar her:* $\dots$

# Godt gået!

I kan nu: one-hot-kode kategorier, bruge sklearn's fit/predict/score-mønster, læse et
beslutningstræ, aflæse feature importance, og forklare hvorfor 100 træer slår ét på
støjede data — og hvorfor træ-modeller er førstevalget på tabeldata.

**Værktøjskassen indtil videre:** tabeldata → træer/skove · billeder, sekvenser, sprog
→ neurale netværk (se de andre notebooks!).